In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
CREATE WIDGET TEXT storageName DEFAULT "adlsproyecto2404"

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

## EXTERNAL LOCATIONS

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para el metastore del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

## CATALOGO

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_clinica
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo principal para el proyecto de clinica ETL'

## ESQUEMAS

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_clinica.raw;
CREATE SCHEMA IF NOT EXISTS catalog_clinica.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_clinica.silver;
CREATE SCHEMA IF NOT EXISTS catalog_clinica.golden;

## TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.citas (
id_cita INTEGER,
fecha_programada TIMESTAMP,
id_paciente INTEGER,
id_medico INTEGER,
id_especialidad INTEGER,
estado STRING,
motivo_cancelacion STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/citas"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.pacientes (
id_paciente INTEGER,
nombre_completo STRING,
sexo STRING,
edad INTEGER,
fecha_nacimiento DATE,
distrito STRING,
tipo_seguro STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/pacientes"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.medicos (
id_medico INTEGER,
nombre_completo STRING,
sexo STRING,
id_especialidad INTEGER,
turno STRING,
anos_experiencia INTEGER,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/medicos"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.especialidades (
id_especialidad INTEGER,
nombre_especialidad STRING,
area STRING,
costo_consulta_soles INTEGER,
duracion_min INTEGER,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/especialidades"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.atenciones (
id_atencion INTEGER,
id_cita INTEGER,
fecha_atencion TIMESTAMP,
id_paciente INTEGER,
id_medico INTEGER,
diagnostico STRING,
tratamiento STRING,
duracion_min INTEGER,
requiere_seguimiento STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/atenciones"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.bronze.pagos (
id_pago INTEGER,
id_atencion INTEGER,
id_cita INTEGER,
monto_total INTEGER,
descuento INTEGER,
monto_pagado INTEGER,
tipo_pago STRING,
estado_pago STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/pagos"

## TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.silver.citas_detalle (
id_cita INTEGER,
fecha_programada TIMESTAMP,
anio INTEGER,
mes INTEGER,
dia_semana STRING,
hora INTEGER,
turno STRING,
id_paciente INTEGER,
nombre_paciente STRING,
sexo_paciente STRING,
edad_paciente INTEGER,
distrito_paciente STRING,
tipo_seguro STRING,
id_medico INTEGER,
nombre_medico STRING,
nombre_especialidad STRING,
area_especialidad STRING,
estado STRING,
motivo_cancelacion STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/citas_detalle"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.silver.atenciones_detalle (
id_atencion INTEGER,
id_cita INTEGER,
fecha_atencion TIMESTAMP,
id_paciente INTEGER,
nombre_paciente STRING,
distrito_paciente STRING,
tipo_seguro STRING,
id_medico INTEGER,
nombre_medico STRING,
nombre_especialidad STRING,
diagnostico STRING,
tratamiento STRING,
duracion_min INTEGER,
requiere_seguimiento STRING,
monto_pagado INTEGER,
tipo_pago STRING,
estado_pago STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/atenciones_detalle"

## TABLAS GOLDEN

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.golden.desercion_por_especialidad (
nombre_especialidad STRING,
area_especialidad STRING,
total_citas INTEGER,
total_atendidos INTEGER,
total_inasistencias INTEGER,
total_cancelados INTEGER,
tasa_desercion DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/desercion_por_especialidad"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.golden.desercion_por_seguro (
tipo_seguro STRING,
total_citas INTEGER,
total_inasistencias INTEGER,
tasa_desercion DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/desercion_por_seguro"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.golden.ingresos_por_especialidad (
nombre_especialidad STRING,
area_especialidad STRING,
total_atenciones INTEGER,
ingreso_total DOUBLE,
ingreso_promedio DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/ingresos_por_especialidad"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_clinica.golden.pacientes_recurrentes (
id_paciente INTEGER,
nombre_paciente STRING,
distrito STRING,
tipo_seguro STRING,
total_citas INTEGER,
total_atenciones INTEGER,
total_inasistencias INTEGER,
tasa_desercion DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/pacientes_recurrentes"